## EnTex Spliser Analysis

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

# Set display options
pd.options.display.float_format = '{:.6f}'.format

ModuleNotFoundError: No module named 'seaborn'

In [ ]:
# ENTER PATHS HERE
IN_FILE = "path/to/diffSpliSE_output.tsv"
TARGET_FILE = "path/to/targets.tsv"

# Read in data
spliser_out = pd.read_csv(IN_FILE, sep='\t')
targets = pd.read_csv(TARGET_FILE, sep='\t')

# Prepare grouping variables
group = targets['Group'].astype('category')
group2 = np.repeat(group.values, 2)
lib_size2 = np.repeat(targets['Library_size'].values, 2)

print(targets.head())

# mapq_gte_mapq_cut_unique

In [ ]:
# Filter NA containing Rows (not including the Gene Column)
spliser_narm = spliser_out.dropna(subset=[c for c in spliser_out.columns if c != 'Gene'])

# Filter sites based on SSE (Splice Site Efficiency) range
# SSE = alpha / (alpha + beta)
sse_cols = [c for c in spliser_narm.columns if "_SSE" in c]
mean_sse = spliser_narm[sse_cols].mean(axis=1)

spliser_filt = spliser_narm[(mean_sse > 0.05) & (mean_sse < 0.95)].copy()
print(f"Remaining sites after filtering: {len(spliser_filt)}")

In [ ]:
# Select alpha and beta columns
count_cols = [c for c in spliser_filt.columns if "_alpha" in c or "_beta" in c]
counts = spliser_filt[count_cols].values.T  # Transpose for PyDESeq2 (Samples as rows)

# Create metadata for the counts
eff = np.tile(["alpha", "beta"], len(targets))
sample_ids = np.repeat(targets['Sample'].values, 2)

metadata = pd.DataFrame({
    "Sample": sample_ids,
    "Eff": eff,
    "Group": group2,
    "Combo": [f"{e}.{g}" for e, g in zip(eff, group2)]
}, index=[f"S{i}_{e}" for i, e in enumerate(eff)])

# Initialize PyDESeq2 object
dds = DeseqDataSet(
    counts=pd.DataFrame(counts, index=metadata.index, columns=spliser_filt.index),
    metadata=metadata,
    design_factors="Combo",
    refit_cooks=True
)

In [ ]:
# Run the DESeq2 pipeline (Estimates dispersions and fits GLM)
dds.deseq2()

print(f"Dispersion: {dds.dispersions_[:5]}")

In [ ]:
# Define the contrast for the interaction
# Note: Adjust these strings based on your actual Group names in the targets file
contrast = ["Combo", "alpha.Col", "beta.Col", "alpha.SC35.scl", "beta.SC35.scl"]

stat_res = DeseqStats(dds, contrast=contrast)
stat_res.summary()

# Get all results and significant hits
all_results = stat_res.results_df
hits = all_results[all_results['padj'] < 0.05]

In [ ]:
# Merge back with original site labels
hits_intersect = hits.join(spliser_filt[['Region', 'Site', 'Gene', 'Strand'] + sse_cols])

# Calculate averages for Delta SSE
# Update these column filters based on your specific condition names
cond1_cols = [c for c in sse_cols if "Col" in c] # Control
cond2_cols = [c for c in sse_cols if "sc35" in c] # Test

hits_intersect['avg1'] = hits_intersect[cond1_cols].mean(axis=1)
hits_intersect['avg2'] = hits_intersect[cond2_cols].mean(axis=1)
hits_intersect['dSSE'] = hits_intersect['avg2'] - hits_intersect['avg1']

# Filter by magnitude of change
hits_sse_filtered = hits_intersect[hits_intersect['dSSE'].abs() >= 0.1].sort_values('padj')

In [ ]:
# Export to TSV
hits_intersect.to_csv("All_Python.tsv", sep='\t', index=False)
hits_sse_filtered.to_csv("BHhits_SSEFiltered_Python.tsv", sep='\t', index=False)

print("Analysis Complete. Files saved.")